In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [ ]:
credit_df = pd.read_csv('./data/train.csv')
credit_df.head()

In [ ]:
credit_df.count()

In [ ]:
credit_df.nunique(axis=0)

In [ ]:
credit_df.isnull().sum()

In [ ]:
credit_df.info()

In [ ]:
credit_df.describe()

In [ ]:
credit_df.shape

# Data Preprocessing

-Handle missing values in name column by looking up customer_id and using the name that belongs to the corresping id

In [ ]:
all_Customer_ID = credit_df[credit_df['Name'].isnull()]['Customer_ID'].values
print(type(all_Customer_ID))
print(all_Customer_ID.shape)
#print(len(all_Customer_ID))
for id in all_Customer_ID:
  name = credit_df.loc[(credit_df['Customer_ID'] == id) & (credit_df['Name'].notna())]['Name'].values[0]
  credit_df.loc[(credit_df['Customer_ID'] == id) & (credit_df['Name'].isna()), ['Name']] = name

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0xd40']['Name']

-Remove any additional characters in the age field i.e. '_', convert age field to type int, replace abnormal ages with the correct age of the person, looking up corresponding customer_id

In [ ]:
#credit_df['Age'] = [str(s).replace('_', '') for s in credit_df['Age']]
#for val in credit_df['Age']:
#  credit_df['Age'] = str(val).replace('_', '')
credit_df['Age'] = [str(val).replace('_', '') for val in credit_df['Age']]
credit_df['Age'] = credit_df['Age'].astype(int)
abnormal_ages= credit_df[(credit_df['Age'] < 10) | (credit_df['Age'] > 70)]['Customer_ID'].values #id or abnoraml ages
print(abnormal_ages.shape)
for id in abnormal_ages:
    age = credit_df[credit_df['Customer_ID'] ==id]['Age'].drop_duplicates().values[0]
    if age < 10 or age > 70:
        age = credit_df[credit_df['Customer_ID'] ==id]['Age'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & ((credit_df['Age'] < 10) | (credit_df['Age'] > 70)), ['Age']] = age

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0xd40']['Age']

In [ ]:
credit_df['Occupation'].unique()

Replace occupation of '______' with the correct occupation of the applicant by looking up the corresponding customer_id

In [ ]:
print(credit_df['Occupation'].isnull().sum())
credit_df[credit_df['Occupation'] == '_______']

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x4d43']['Occupation']

In [ ]:
empty_occupations = credit_df[credit_df['Occupation']== '_______']['Customer_ID'].values
for id in empty_occupations:
  occupation = credit_df[credit_df['Customer_ID'] ==id]['Occupation'].drop_duplicates().values[0]
  if occupation == '_______':
    occupation = credit_df[credit_df['Customer_ID'] ==id]['Occupation'].drop_duplicates().values[1]
  credit_df.loc[(credit_df['Customer_ID'] == id) & (credit_df['Occupation'] == '_______'), ['Occupation']] = occupation

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x4d43']['Occupation']

Remove any additional characters in the annual_income field
i.e. '_', convert annual_income field to type float, replcae annual_income outliers by using the correct annual income from the corresponding customer_id

In [ ]:
credit_df['Annual_Income'] = [str(val).replace('_', '') for val in credit_df['Annual_Income']]
credit_df['Annual_Income'] = credit_df['Annual_Income'].astype(float)

In [ ]:
plt.boxplot(credit_df['Annual_Income'])
plt.show()

In [ ]:
#credit_df.nlargest(5000, ['Annual_Income'])['Customer_ID']
credit_df[credit_df['Annual_Income'] > 500000]['Customer_ID']
credit_df[credit_df['Annual_Income'] < 5000]['Customer_ID']

In [ ]:
outlier_salary = credit_df[credit_df['Annual_Income']> 100000]['Customer_ID'].values
for id in outlier_salary:
    income = credit_df[credit_df['Customer_ID'] ==id]['Annual_Income'].min()
    credit_df.loc[(credit_df['Customer_ID'] ==id) &
              (credit_df['Annual_Income']> 100000), ['Annual_Income']] = income

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x284a']['Annual_Income']

In [ ]:
plt.boxplot(credit_df['Annual_Income'])
plt.show()

Replace null values with the correct monthly salary by looking up the corresponding customer_id

In [ ]:
print('Null monthly salary count = ',credit_df['Monthly_Inhand_Salary'].isnull().sum())
credit_df['Monthly_Inhand_Salary'].replace(np.nan, -1.0, inplace=True)
null_msalary = credit_df[credit_df['Monthly_Inhand_Salary']==-1.0]['Customer_ID'].values
for id in null_msalary:
    salary = credit_df[credit_df['Customer_ID'] == id]['Monthly_Inhand_Salary'].drop_duplicates().values[0]
  #salary=credit_df[credit_df['Customer_ID'] ==
   #               id]['Monthly_Inhand_Salary'].drop_duplicates().sort_values().values[-1]
    if salary == -1.0:
        salary = credit_df[credit_df['Customer_ID'] == id]['Monthly_Inhand_Salary'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & (credit_df['Monthly_Inhand_Salary'] == -1.0), ['Monthly_Inhand_Salary']] = salary

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0xd40']['Monthly_Inhand_Salary']

In [ ]:
print('Null monthly salary count = ',credit_df['Monthly_Inhand_Salary'].isnull().sum())
print(credit_df[credit_df['Monthly_Inhand_Salary'] == -1.0].size)

Replace abnormal bank account values with the correct value for the applicant by looking up corresponding customer_id

In [ ]:
len(credit_df[(credit_df['Num_Bank_Accounts'] > 7)])

In [ ]:
len(credit_df[(credit_df['Num_Bank_Accounts'] < 0)])

In [ ]:
credit_df[(credit_df['Num_Bank_Accounts'] > 7) |
        (credit_df['Num_Bank_Accounts'] < 0)]

In [ ]:
abnormal_accounts = credit_df[(credit_df['Num_Bank_Accounts'] > 7) |(credit_df['Num_Bank_Accounts'] < 0)]['Customer_ID'].values

for id in abnormal_accounts:
    numAccounts = credit_df[credit_df['Customer_ID'] == id]['Num_Bank_Accounts'].drop_duplicates().values[0]
    if numAccounts < 0 or numAccounts > 11:
        numAccounts = credit_df[credit_df['Customer_ID'] == id]['Num_Bank_Accounts'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & (
        (credit_df['Num_Bank_Accounts'] < 0) | (credit_df['Num_Bank_Accounts'] > 11)), ['Num_Bank_Accounts']] = numAccounts

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x4004']['Num_Bank_Accounts']

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x96ca']['Num_Bank_Accounts']

In [ ]:
print('Null credit card count = ',credit_df['Num_Credit_Card'].isnull().sum())

In [ ]:
credit_df['Num_Credit_Card'] = [str(val).replace('_', '') for val in credit_df['Num_Credit_Card']]
credit_df['Num_Credit_Card'] = credit_df['Num_Credit_Card'].astype(int)

In [ ]:
#credit_df[(credit_df['Num_of_Loan'] > 10) |(credit_df['Num_of_Loan'] <= 0)]
print(credit_df[credit_df['Num_Credit_Card'] < 0].size)
credit_df[credit_df['Customer_ID'] == 'CUS_0x4100']['Num_Credit_Card']

In [ ]:
abnormal_cc = credit_df[credit_df['Num_Credit_Card'] > 7]['Customer_ID'].values
#print(abnormal_cc.size)
for id in abnormal_cc:
    #print(credit_df[credit_df['Customer_ID'] == id]['Num_Credit_Card'])
    cc = credit_df[credit_df['Customer_ID'] == id]['Num_Credit_Card'].drop_duplicates().values[0]
    if cc > 15:
        cc = credit_df[credit_df['Customer_ID'] == id]['Num_Credit_Card'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & (
        (credit_df['Num_Credit_Card'] < 0) | (credit_df['Num_Credit_Card'] > 11)), ['Num_Credit_Card']] = cc

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x9bc1']['Num_Credit_Card']

In [ ]:
credit_df['Interest_Rate'] = [str(val).replace('_', '') for val in credit_df['Interest_Rate']]
credit_df['Interest_Rate'] = credit_df['Interest_Rate'].astype(int)

In [ ]:
print(credit_df[credit_df['Interest_Rate'] <= 0].size)
print('Null interest rate count = ',credit_df['Interest_Rate'].isnull().sum())

In [ ]:
credit_df[credit_df['Customer_ID'] == 'CUS_0x56c5']['Interest_Rate']
credit_df[credit_df['Customer_ID'] == 'CUS_0xc365']['Interest_Rate']


In [ ]:
print(credit_df[credit_df['Interest_Rate'] > 100]['Customer_ID'].values.size)
print(credit_df[credit_df['Interest_Rate'] > 100]['Customer_ID'].unique().size)
print(credit_df[credit_df['Customer_ID'] == 'CUS_0x1915']['Interest_Rate'])

In [ ]:
abnormal_interest = credit_df[credit_df['Interest_Rate'] > 100]['Customer_ID'].values
for id in abnormal_interest:
  if credit_df[credit_df['Customer_ID'] == id]['Interest_Rate'].drop_duplicates().values.size > 1:
    rate = credit_df[credit_df['Customer_ID'] == id]['Interest_Rate'].drop_duplicates().values[0]
    if rate > 100:
          rate = credit_df[credit_df['Customer_ID'] == id]['Interest_Rate'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & (
        (credit_df['Interest_Rate'] < 0) | (credit_df['Interest_Rate'] > 11)), ['Interest_Rate']] = rate
  #else:
    #print(credit_df[credit_df['Customer_ID'] == id]['Interest_Rate'])
mean_interest = credit_df[credit_df['Interest_Rate'] <= 100]['Interest_Rate'].values

In [ ]:
print(credit_df[credit_df['Customer_ID'] == 'CUS_0xac67']['Interest_Rate'])

In [ ]:
credit_df['Num_of_Loan'] = [str(val).replace('_', '') for val in credit_df['Num_of_Loan']]
credit_df['Num_of_Loan'] = credit_df['Num_of_Loan'].astype(int)

In [ ]:
credit_df[(credit_df['Num_of_Loan'] > 10) |(credit_df['Num_of_Loan'] <= 0)]

In [ ]:
abnormal_loan = credit_df[(credit_df['Num_of_Loan'] > 9) | (
    credit_df['Num_of_Loan'] <= 0)]['Customer_ID'].values

for id in abnormal_loan:
    numLoans = credit_df[credit_df['Customer_ID'] == id]['Num_of_Loan'].drop_duplicates().values[0]
    if numLoans < 0 or numLoans > 9:
        numLoans = credit_df[credit_df['Customer_ID'] == id]['Num_of_Loan'].drop_duplicates().values[1]
    credit_df.loc[(credit_df['Customer_ID'] == id) & (
        (credit_df['Num_of_Loan'] < 0) | (credit_df['Num_of_Loan'] > 9)), ['Num_of_Loan']] = numLoans

In [ ]:
print(credit_df[credit_df['Customer_ID'] == 'CUS_0x9322']['Num_of_Loan'])
print(credit_df[credit_df['Customer_ID'] == 'CUS_0x4e53']['Num_of_Loan'])

In [ ]:
plt.hist(credit_df['Credit_Score'])
plt.show()

In [ ]:
X_df = credit_df.drop('Credit_Score', axis=1)
corr_matrix = X_df.corr().abs()
sns.heatmap(corr_matrix)

In [ ]:
data = credit_df.groupby(['Payment_of_Min_Amount','Credit_Score'],sort=False,as_index=False).count()
sns.barplot(data=data,x='Credit_Score',y='ID',hue='Payment_of_Min_Amount').set_title('Payment of Minimum Amount vs Credit Score')
plt.ylabel('Count')
plt.show()

In [ ]:
# Shumail

In [ ]:
# df = pd.read_csv('train.csv')

In [ ]:
df = credit_df # for consistent changes on the same data frames

In [ ]:
df.count()

In [ ]:
df.info()

In [ ]:
df.nunique(axis=0)

In [ ]:
df.isnull().sum()

In [ ]:
# feature 13: Type_of_Loan (Represents the average number of days delayed from the payment date)

In [ ]:
t = df.loc[df['Type_of_Loan'].isna(), ['Customer_ID', 'Type_of_Loan']]

In [ ]:
r = t.groupby('Customer_ID')['Type_of_Loan'].agg(lambda x: x.isna().sum() != 8)
r[r].reset_index()

In [ ]:
# these missing values are empty for all the 8 months of customer ids and hence, cannot be computed.

In [ ]:
df['Type_of_Loan'].value_counts()[:5] # replacing missing values with the most popular category: Not Specified

In [ ]:
df['Type_of_Loan'].fillna('Not Specified', inplace=True)

In [ ]:
# feature 14: Delay_from_due_date (Represents the average number of payments delayed by a person)

In [ ]:
df['Delay_from_due_date'].unique()

In [ ]:
df['Delay_from_due_date'].mean() # mean delay days in payments

In [ ]:
# feature 15: Num_of_Delayed_Payment (Represents the percentage change in credit card limit)

In [ ]:
df['Num_of_Delayed_Payment'].unique()

In [ ]:
df['Num_of_Delayed_Payment'] = df['Num_of_Delayed_Payment'].str.replace(r'(\d+)_', r'\1', regex=True) # remove value_ with value

In [ ]:
arr = df['Num_of_Delayed_Payment'].unique()
non_digit_values = [str(value) for value in arr if re.search(r'\D', str(value))]

for value in non_digit_values:
    print(value)

In [ ]:
t1 = df.loc[df['Num_of_Delayed_Payment'].isna(), ['Customer_ID', 'Num_of_Delayed_Payment']]

In [ ]:
r = t1.groupby('Customer_ID')['Num_of_Delayed_Payment'].agg(lambda x: x.isna().sum() == 8)
r[r].reset_index()

In [ ]:
df['Num_of_Delayed_Payment'] = pd.to_numeric(df['Num_of_Delayed_Payment'], errors='coerce') # convert to numeric

In [ ]:
df['Num_of_Delayed_Payment'] = np.where(df['Num_of_Delayed_Payment'] < 0, np.nan, df['Num_of_Delayed_Payment'])

In [ ]:
result = df.groupby('Customer_ID')['Num_of_Delayed_Payment'].mean()
result

In [ ]:
for i in range(len(df['Num_of_Delayed_Payment'])):
  if pd.isna(df.at[i, 'Num_of_Delayed_Payment']):
    df.loc[i, 'Num_of_Delayed_Payment'] = result[df['Customer_ID'][i]].copy()

In [ ]:
df['Num_of_Delayed_Payment'].unique()[:40]

In [ ]:
df['Num_of_Delayed_Payment'].mean() # mean number of payments delayed

In [ ]:
# feature 16: Changed_Credit_Limit (Represents the percentage change in credit card limit)

In [ ]:
df['Changed_Credit_Limit'].head(15)

In [ ]:
df['Changed_Credit_Limit'] = df['Changed_Credit_Limit'].replace('_', np.nan)
df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'], errors='coerce') # convert to numeric

In [ ]:
df[df['Changed_Credit_Limit'] < 0]

In [ ]:
df['Changed_Credit_Limit'][pd.isna(df['Changed_Credit_Limit'])]

In [ ]:
t2 = df.loc[df['Changed_Credit_Limit'].isna(), ['Customer_ID', 'Changed_Credit_Limit']]

In [ ]:
r = t2.groupby('Customer_ID')['Changed_Credit_Limit'].agg(lambda x: x.isna().sum() == 8)
r[r].reset_index()

In [ ]:
result = df.groupby('Customer_ID')['Changed_Credit_Limit'].mean()
result

In [ ]:
for i in range(len(df['Changed_Credit_Limit'])):
  if pd.isna(df.at[i, 'Changed_Credit_Limit']):
    df.loc[i, 'Changed_Credit_Limit'] = result[df['Customer_ID'][i]].copy()

In [ ]:
df['Changed_Credit_Limit'].unique()[:40]

In [ ]:
df['Changed_Credit_Limit'].mean() # mean % change in credit card limit

In [ ]:
# feature 17: Num_Credit_Inquiries (Represents the number of credit card inquiries)

In [ ]:
df['Num_Credit_Inquiries'].unique()

In [ ]:
df[df['Num_Credit_Inquiries'] < 0]

In [ ]:
df['Num_Credit_Inquiries'][pd.isna(df['Num_Credit_Inquiries'])]

In [ ]:
t3 = df.loc[df['Num_Credit_Inquiries'].isna(), ['Customer_ID', 'Num_Credit_Inquiries']]

In [ ]:
r = t3.groupby('Customer_ID')['Num_Credit_Inquiries'].agg(lambda x: x.isna().sum() == 8)
r[r].reset_index()

In [ ]:
result = df.groupby('Customer_ID')['Num_Credit_Inquiries'].mean()
result

In [ ]:
for i in range(len(df['Num_Credit_Inquiries'])):
  if pd.isna(df.at[i, 'Num_Credit_Inquiries']):
    df.loc[i, 'Num_Credit_Inquiries'] = result[df['Customer_ID'][i]].copy()

In [ ]:
df['Num_Credit_Inquiries'].unique()[:40]

In [ ]:
df['Num_Credit_Inquiries'].mean() # mean number of credit card inquiries

In [ ]:
# feature 18: Credit_Mix (Represents the classification of the mix of credits)

In [ ]:
df.loc[:,['Customer_ID', 'Credit_Mix']].head(15)

In [ ]:
df['Credit_Mix'].unique()

In [ ]:
df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)

In [ ]:
result = df.groupby('Customer_ID')['Credit_Mix'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
result

In [ ]:
for i in range(len(df['Credit_Mix'])):
  if pd.isna(df.at[i, 'Credit_Mix']):
    df.loc[i, 'Credit_Mix'] = result[df['Customer_ID'][i]]

In [ ]:
# feature 19: Outstanding_Debt (Represents the remaining debt to be paid (in USD))

In [ ]:
arr = df['Outstanding_Debt'].unique()
arr

In [ ]:
for value in arr:
    if value.endswith('_'):
        print(value)

In [ ]:
df['Outstanding_Debt'] = df['Outstanding_Debt'].str.replace(r'(\d+)_', r'\1', regex=True) # remove value_ with value

In [ ]:
df['Outstanding_Debt'] = pd.to_numeric(df['Outstanding_Debt'], errors='coerce') # convert to numeric

In [ ]:
df[df['Outstanding_Debt'] < 0]

In [ ]:
df['Outstanding_Debt'][pd.isna(df['Num_Credit_Inquiries'])]

In [ ]:
df['Outstanding_Debt'].mean() # 1426 USD outstanding debt on average

In [ ]:
# feature 20: Credit_Utilization_Ratio (Represents the utilization ratio of credit card)

In [ ]:
df['Credit_Utilization_Ratio']

In [ ]:
df[df['Credit_Utilization_Ratio'] < 0]

In [ ]:
df['Credit_Utilization_Ratio'].nunique()

In [ ]:
df['Credit_Utilization_Ratio'].mean() # average utilization ratio of credit card is 32

In [ ]:
# feature 21: Credit_History_Age

In [ ]:
df['Credit_History_Age'].head(10)

In [ ]:
df.loc[:,['Customer_ID', 'Credit_History_Age']].head(30)

In [ ]:
years_months = df["Credit_History_Age"].str.extract(r'(\d+)\s*Years?\s*and\s*(\d+)\s*Months?')

In [ ]:
years_months.head(15)

In [ ]:
years_months.loc[1, 0]

In [ ]:
for col in years_months.columns:
    years_months[col] = pd.to_numeric(years_months[col], errors='coerce')

In [ ]:
for i in range(len(years_months[0])):
  if pd.isna(years_months.at[i, 1]):
    if i%8 == 0:
      years_months.loc[i, 1] = 1
    else:
      years_months.loc[i, 1] = years_months.loc[i-1, 1] + 1

In [ ]:
years_months[years_months.isna().any(axis=1)]

In [ ]:
years_months['Customer_ID'] = df['Customer_ID']

In [ ]:
result = years_months.groupby('Customer_ID')[0].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
result

In [ ]:
for i in range(len(years_months[0])):
  if pd.isna(years_months.at[i, 0]):
    years_months.loc[i, 0] = result[years_months['Customer_ID'][i]]

In [ ]:
years_months[2] = (years_months[0] * 12) + years_months[1]
years_months

In [ ]:
df['Credit_History_Age'] = years_months[2]

In [ ]:
df['Credit_History_Age'].mean() # 221 months is the average age of credit history

In [ ]:
# feature 22: Payment_of_Min_Amount (Represents whether only the minimum amount was paid by the person)

In [ ]:
df['Payment_of_Min_Amount']

In [ ]:
df['Payment_of_Min_Amount'].unique()

In [ ]:
df['Payment_of_Min_Amount'].value_counts()

In [ ]:
# feature 23: Total_EMI_per_month (Represents the monthly EMI payments (in USD))

In [ ]:
df['Total_EMI_per_month']

In [ ]:
df[df['Total_EMI_per_month'] < 0]

In [ ]:
df['Total_EMI_per_month'].unique()[:100]

In [ ]:
df['Total_EMI_per_month'].mean() # average monthly EMI payments (in USD) are 1403

In [ ]:
# feature 24: Amount_invested_monthly (Represents the monthly amount invested by the customer (in USD))

In [ ]:
df['Amount_invested_monthly'].unique()[:200]

In [ ]:
df['Amount_invested_monthly'] = df['Amount_invested_monthly'].str.replace(r'_', '')
df['Amount_invested_monthly'] = pd.to_numeric(df['Amount_invested_monthly'], errors='coerce') # convert to numeric

In [ ]:
df[df['Amount_invested_monthly'] < 0]

In [ ]:
t4 = df.loc[df['Amount_invested_monthly'].isna(), ['Customer_ID', 'Amount_invested_monthly']]

In [ ]:
r = t4.groupby('Customer_ID')['Amount_invested_monthly'].agg(lambda x: x.isna().sum() == 8)
r[r].reset_index()

In [ ]:
result = df.groupby('Customer_ID')['Amount_invested_monthly'].mean()
result

In [ ]:
for i in range(len(df['Amount_invested_monthly'])):
  if pd.isna(df.at[i, 'Amount_invested_monthly']):
    df.loc[i, 'Amount_invested_monthly'] = result[df['Customer_ID'][i]].copy()

In [ ]:
df['Amount_invested_monthly'].isna().sum()

In [ ]:
df['Amount_invested_monthly'].mean() # average monthly amount invested is 638 USD

In [ ]:
# feature 25: Payment_Behaviour (Represents the payment behavior of the customer (in USD))

In [ ]:
df['Payment_Behaviour']

In [ ]:
df['Payment_Behaviour'].value_counts()

In [ ]:
df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', 'Not_specified')

In [ ]:
# feature 26: Monthly_Balance (Represents the monthly balance amount of the customer (in USD))

In [ ]:
df['Monthly_Balance']

In [ ]:
df['Monthly_Balance'].unique()[300:500]

In [ ]:
df[df['Monthly_Balance'].isna()]

In [ ]:
df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'], errors='coerce') # convert to numeric

In [ ]:
df[df['Monthly_Balance'] < 0]

In [ ]:
t5 = df.loc[df['Monthly_Balance'].isna(), ['Customer_ID', 'Monthly_Balance']]

In [ ]:
r = t5.groupby('Customer_ID')['Monthly_Balance'].agg(lambda x: x.isna().sum() == 8)
r[r].reset_index()

In [ ]:
result = df.groupby('Customer_ID')['Monthly_Balance'].mean()
result

In [ ]:
for i in range(len(df['Monthly_Balance'])):
  if pd.isna(df.at[i, 'Monthly_Balance']):
    df.loc[i, 'Monthly_Balance'] = result[df['Customer_ID'][i]].copy()

In [ ]:
df['Monthly_Balance'].mean() # average monthly balance is 403 USD

In [ ]:
# feature 27: Credit_Score ()

In [ ]:
df['Credit_Score'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.count()

In [ ]:
df.info()

In [ ]:
# Preprocessing features from 13 to 27

In [ ]:
# encoding response variable
mapping = {'Standard': 0.5, 'Poor': 0, 'Good': 1}
df['Credit_Score'] = df['Credit_Score'].replace(mapping)
df['Credit_Score'].value_counts()

In [ ]:
# dropping columns no longer needed for modeling phase

In [ ]:
df['Type_of_Loan'].unique()[:20]

In [ ]:
len(df['Type_of_Loan'].unique())

In [ ]:
new_df = df.drop(['ID', 'Customer_ID', 'Name', 'SSN', 'Type_of_Loan'], axis=1)

In [ ]:
new_df.info()

In [ ]:
new_df['Month'].value_counts()

In [ ]:
new_df['Payment_of_Min_Amount'].value_counts()

In [ ]:
new_df['Payment_Behaviour'].value_counts()

In [ ]:
new_df['Occupation'].value_counts()

In [ ]:
new_df['Credit_Mix'].value_counts()

In [ ]:
# encoding all categorical features as one-hot-encoding except for credit_mix which will be ordinally encoded

new_df =  pd.get_dummies(new_df, columns = ['Month', 'Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour'], drop_first=True)
new_df

In [ ]:
quality_order = ['Bad', 'Standard', 'Good']
encoder = OrdinalEncoder(categories=[quality_order])
encoder.fit(new_df[['Credit_Mix']])

new_df['Credit_Mix'] = encoder.transform(new_df[['Credit_Mix']].values.reshape(-1, 1))
new_df['Credit_Mix'].value_counts()

In [ ]:
# splitting df into df_X and df_Y
df_X = new_df.drop(columns=['Credit_Score'])
df_Y = new_df['Credit_Score']

In [ ]:
# correlation matrices and dropping more features with correlations > 0.9

In [ ]:
df_X.corr()

In [ ]:
corr = df_X.corr().abs()
utm = np.triu(np.ones(corr.shape), k=1).astype(bool)
upper = corr.where(utm)

to_drop = [column for column in upper.columns if any(upper[column] >= 0.90)]
print('Columns dropped: ', to_drop) # monthly_inhand_salary has a correlation of 0.998237 with annual_income --> hence, dropped.
df_X.drop(to_drop, axis=1, inplace=True)

df_X

In [ ]:
# Model training